In [ ]:
# SPFC MANUAL VFA ABLATION — EDIT THIS CELL FIRST
# Generation is deliberately disabled by default. Review the plan, then set this True.
RUN_GENERATION = True
OVERWRITE_EXISTING = True
RUN_TWO_FACTOR_GRID = False
RUN_FINALIST_VALIDATION = False
SAVE_MANUAL_RATINGS = False
RUN_STANDALONE_PRIMITIVE_GALLERY = True
RUN_PRIMITIVE_WEIGHT_SWEEP = False
RUN_PRIMITIVE_DUPLICATION_TEST = False

# Paths
CONFIG_PATH = 'configs/sd3_medium_kaggle.yaml'
OUTPUT_ROOT = 'outputs/spfc_manual_vfa_ablation'

# Frozen model / sampler controls. Keep fixed while comparing SPFC settings.
MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
DTYPE = 'float16'
LOAD_T5_TEXT_ENCODER = False
ENABLE_MODEL_CPU_OFFLOAD = True
ENABLE_VAE_SLICING = True
HEIGHT = 1024
WIDTH = 1024
NUM_INFERENCE_STEPS = 24
GUIDANCE_SCALE = 4.5
SEEDS = [42]                 # Add seeds only after narrowing the search.
DISPLAY_SEED = 42

# The notebook contains all five prompts. Reduce these lists for cheap smoke tests.
ALL_PROMPT_KEYS = [
    'kitchen_scene',
    # 'white_dove_black_feather',
    # 'marble_fox_beach_snow',
    # 'purple_elephant_green_microwave',
    # 'blue_bench_red_ball',
]
ACTIVE_PROMPT_KEYS = ALL_PROMPT_KEYS[:]
SWEEP_PROMPT_KEYS = ['kitchen_scene']  # Expand to ALL_PROMPT_KEYS after a useful range emerges.
PRIMITIVE_GALLERY_PROMPT_KEYS = ACTIVE_PROMPT_KEYS[:]
PRIMITIVE_GALLERY_MAX_PRIMITIVES = None  # None means every enabled primitive.
PRIMITIVE_GALLERY_INCLUDE_SOURCE = True
PRIMITIVE_GALLERY_INCLUDE_TARGET = True

# Focused primitive-pressure audit. Names must match the decomposition block below.
PRESSURE_PROMPT_KEY = 'kitchen_scene'
PRESSURE_PRIMITIVE_NAME = 'black_countertop'
PRESSURE_SEED = SEEDS[0]
PRIMITIVE_WEIGHT_VALUES = [0.0, 0.25, 0.50, 1.0, 1.5, 2.0, 3.0]
PRIMITIVE_DUPLICATION_COUNTS = [1, 2, 3]

# Reference SPFC settings. This lists every PrimitiveFlowConfig field.
BASE_SPFC_KNOBS = {
    # Locked routing fields: this notebook calls the sparse primitive sampler directly.
    'enabled': True,
    'mode': 'sparse_primitive_flow',

    # Sparse intervention schedule (zero-based step indices).
    'final_only': False,
    'aggregation_steps': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23],
    'aggregation_step_fractions': None,
    'aggregate_every_n_steps': None,

    # Conditions included in VFA. The target is a locked invariant in this implementation.
    'include_source_flow': True,
    'include_target_flow': True,
    'target_reference': True,
    'max_primitives': 5,

    # VFA scoring and weighting.
    'use_consensus_gating': True,
    'use_target_consistency_gating': True,
    'uniform_condition_weights': False,
    'source_weight': 1.0,
    'target_weight': 1.0,
    'vfa_temperature': 1,
    'min_gate': 0.0,
    'max_gate': 1.5,

    # Target-relative correction controls.
    'velocity_clip_ratio': 0.50,
    'steering_strength': 1.0,

    # Latent Trajectory Projection (second safety bound).
    'ltp_enabled': True,
    'ltp_mode': 'latent',      # 'latent', 'velocity', or 'off'
    'ltp_radius_ratio': 0.35,
    'fallback_to_velocity_ltp': True,

    # Memory behavior; this should not change the intended math.
    'sequential_condition_forward': True,
}

# Named one-component ablations. Select a cheap subset first.
CORE_ABLATIONS = {
    'full_spfc': {},
    'no_aggregation_control': {'aggregation_steps': [], 'aggregation_step_fractions': None, 'aggregate_every_n_steps': None},
    'steering_zero_control': {'steering_strength': 0.0},
    'no_consensus_gate': {'use_consensus_gating': False},
    'no_target_gate': {'use_target_consistency_gating': False},
    'no_gates': {'use_consensus_gating': False, 'use_target_consistency_gating': False},
    'uniform_condition_weights': {'uniform_condition_weights': True},
    'no_source_flow': {'include_source_flow': False},
    'no_velocity_clip': {'velocity_clip_ratio': 1_000_000.0},
    'no_ltp': {'ltp_enabled': False, 'ltp_mode': 'off'},
    'unbounded_correction': {'velocity_clip_ratio': 1_000_000.0, 'ltp_enabled': False, 'ltp_mode': 'off'},
    'dense_all_steps': {'aggregation_steps': None, 'aggregation_step_fractions': None, 'aggregate_every_n_steps': 1},
    'early_only': {'aggregation_steps': [0, 1, 2, 3, 4, 5, 6, 7], 'aggregation_step_fractions': None, 'aggregate_every_n_steps': None},
    'late_only': {'aggregation_steps': [16, 17, 18, 19, 20, 21, 22, 23], 'aggregation_step_fractions': None, 'aggregate_every_n_steps': None},
    'final_only': {'final_only': True},
    'one_primitive_only': {'max_primitives': 1},
    'gate_floor_0p5': {'min_gate': 0.5},
    'gate_ceiling_0p5': {'max_gate': 0.5},
}
ACTIVE_CORE_ABLATIONS = [
    'full_spfc',
    'steering_zero_control',
    'no_consensus_gate',
    'no_target_gate',
    'no_velocity_clip',
    'no_ltp',
]

# One-factor-at-a-time sweeps. Change ACTIVE_SWEEP_KNOB to select a row.
ONE_FACTOR_SWEEPS = {
    'vfa_temperature': [0.2, 0.4, 0.7, 1.0, 2.0],
    'min_gate': [0.0, 0.25, 0.5, 0.75],
    'max_gate': [0.25, 0.5, 0.75, 1.0],
    'velocity_clip_ratio': [0.05, 0.10, 0.25, 0.50, 1.0, 1_000_000.0],
    'steering_strength': [0.0, 0.25, 0.50, 0.75, 1.0, 1.25, 1.5],
    'source_weight': [0.0, 0.3, 0.7, 1.0, 1.5],
    'target_weight': [0.5, 0.8, 1.0, 1.2, 1.5, 2.0],
    'ltp_radius_ratio': [0.05, 0.10, 0.20, 0.35, 0.50, 1.0],
    'max_primitives': [1, 2, 3, 4, 5],
}
ACTIVE_SWEEP_KNOB = 'velocity_clip_ratio'

# Optional two-factor grid. This is expensive; it is off by default.
GRID_X_KNOB = 'velocity_clip_ratio'
GRID_X_VALUES = [0.10, 0.50, 1.0]
GRID_Y_KNOB = 'steering_strength'
GRID_Y_VALUES = [0.50, 1.0, 1.5]
GRID_PROMPT_KEYS = ['kitchen_scene']

# Fill this after inspecting sweeps, then enable RUN_FINALIST_VALIDATION.
FINALIST_OVERRIDES = {}

# Human rating fields, all scored 1 (poor) to 5 (excellent).
MANUAL_RATING_METRICS = [
    'target_fidelity',
    'attribute_binding',
    'object_coverage',
    'composition',
    'visual_quality',
]


# Manual SPFC / VFA ablation laboratory

This notebook is designed to answer a specific question: **which parts of Sparse Primitive Flow Composition actually improve the image, and which settings give the best target fidelity without destabilizing the target trajectory?**

The experimental unit is always a matched pair or sweep using the same model, resolution, prompt decomposition, inference steps, guidance scale, and seed. Change one SPFC component at a time first. Only run multi-knob searches after the one-factor results reveal a sensible range.

The five prompts deliberately cover different compositional failure modes:

1. a multi-part kitchen layout with material/color binding;
2. two small objects with opposing colors;
3. material + clothing + scene + weather binding;
4. two unusual object/color bindings;
5. two common objects with distinct colors.

> Start with `RUN_GENERATION = False`. Run all cells once to validate the notebook and inspect the planned experiment count. Then enable generation and execute only the stage you want.


## What every knob means

### Frozen-model and sampler controls

| Knob | Meaning | Experimental advice |
|---|---|---|
| `MODEL_ID`, `DTYPE`, `LOAD_T5_TEXT_ENCODER` | Model checkpoint, numerical precision, and whether SD3's T5 encoder is loaded. | Treat these as nuisance variables and hold them fixed for all SPFC comparisons. Enabling T5 can materially change prompt understanding. |
| `HEIGHT`, `WIDTH` | Output resolution. | Hold fixed. Structure and memory use change with resolution. |
| `NUM_INFERENCE_STEPS` | Number of rectified-flow solver steps. | Hold fixed because aggregation indices refer to this trajectory. |
| `GUIDANCE_SCALE` | Standard classifier-free guidance scale applied to every condition. | Hold fixed while ablating VFA. |
| `SEEDS` | Initial latent noise seeds. | Use one seed while searching; validate finalists over several seeds. |

### Sparse intervention controls

| Knob | Meaning |
|---|---|
| `aggregation_steps` | Explicit zero-based solver steps where all condition velocities are evaluated and composed. A non-empty explicit list takes precedence over fractions and `aggregate_every_n_steps`. |
| `aggregation_step_fractions` | Resolution-independent step locations expressed from `0.0` to `1.0`; used when no explicit list is supplied. |
| `aggregate_every_n_steps` | Dense periodic intervention; the final step is also included. Used when no explicit list is supplied. |
| `final_only` | Overrides the schedule and aggregates only on the last solver step. This is usually too late for structural changes. |

### Conditions and base weights

| Knob | Meaning |
|---|---|
| `include_source_flow` | Adds the simplified scene prompt to VFA. Disable it to test whether the source stabilizes or dilutes the target. |
| primitive `weight` | Per-primitive prior inside the editable decomposition dictionary below. |
| `source_weight`, `target_weight` | Priors for source and target conditions. They multiply the gates before softmax. |
| `uniform_condition_weights` | Forces source, every primitive, and target base weight to `1.0`, overriding all weights above. |
| `max_primitives` | Takes only the first N enabled primitives. Order therefore matters when this is smaller than the decomposition. |

A base weight of zero does **not** remove a condition: `exp(0)` still receives positive softmax mass. Use `include_source_flow=False` to remove the source, and use the primitive's `enabled=False` flag to remove a primitive. Keep each prompt's negative prompt fixed across variants because it is another conditioning variable.

`enabled`, `mode`, `include_target_flow`, and `target_reference` are shown because they exist in the config schema. This notebook directly calls the sparse sampler, so `enabled`/`mode` are routing labels rather than mathematical ablations. The current sampler deliberately guarantees that the full target is present and always uses it as the clipping/LTP reference. Setting either target field false would not produce a clean target-free ablation, so this notebook locks both to their reference values rather than reporting a misleading experiment.

### VFA arbitration

At each aggregation step, VFA scores condition `i` as:

`raw_score_i = base_weight_i × consensus_gate_i × target_gate_i`

and then computes `softmax(raw_score / temperature)`.

| Knob | Meaning |
|---|---|
| `use_consensus_gating` | Downweights a velocity that has low mean positive cosine agreement with the other conditions. |
| `use_target_consistency_gating` | Downweights a velocity that has low positive cosine agreement with the full-target velocity. |
| `vfa_temperature` | Softmax temperature. Lower values concentrate weight on the highest scores; higher values approach a uniform blend. |
| `min_gate`, `max_gate` | Floor/cap applied to gate values. The underlying positive cosine gates already lie in `[0,1]`, so the default `[0,1]` is effectively unclamped. Only a higher floor or lower ceiling changes the gates. |

### Target-relative safety controls

| Knob | Meaning |
|---|---|
| `velocity_clip_ratio` | Clips `v_raw - v_target` to at most this fraction of `||v_target||`. A very large value approximates removing this clamp. |
| `steering_strength` | Multiplies the correction **after** velocity clipping. Therefore the pre-LTP bound is approximately `steering_strength × velocity_clip_ratio × ||v_target||`. Zero is a target-only control. |
| `ltp_enabled` | Enables the second safety mechanism, Latent Trajectory Projection. |
| `ltp_mode` | `latent` projects the scheduler-produced next latent; `velocity` projects the velocity before stepping; `off` disables LTP. |
| `ltp_radius_ratio` | Maximum candidate offset relative to the norm of the target's own step. A smaller value holds the sample closer to the target trajectory. |
| `fallback_to_velocity_ltp` | Uses velocity-space LTP if the scheduler cannot safely replay target and candidate steps. |

`sequential_condition_forward` changes memory behavior, not the intended VFA equation. Keep it enabled on constrained GPUs.


## Editable prompt decompositions

Edit this block before treating results as final. Each primitive should be a complete sentence that emphasizes one requirement while retaining enough shared scene context to produce a comparable velocity. The examples are starting hypotheses, not ground truth.

A blank template is included at the bottom. Primitive list order matters when `max_primitives` truncates the set.


In [ ]:
# EDIT HERE: replace any source prompt, primitive text, weight, or enabled flag.
PROMPT_DECOMPOSITIONS = {
    'kitchen_scene': {
        'name': 'Two-tone kitchen countertops and sink',
        'target_prompt': (
            'A realistic photo of a modern kitchen scene with a black countertop on one side, '
            'a whitish countertop on the other side, and a gray sink with a tall long-neck faucet '
            'between them, sharp focus'
        ),
        'source_prompt': (
            'two different countertops and a sink with a faucet in a kitchen'
        ),
        'negative_prompt': 'blurry, low quality, distorted kitchen, missing sink, duplicate faucet, text, watermark',
        'primitive_prompts': [
            {'name': 'black_countertop', 'text': 'a black countertop and another countertop and a sink with a faucet in a kitchen, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'whitish_countertop', 'text': 'a whitish countertop and another countertop and a sink with a faucet in a kitchen, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'gray_sink', 'text': 'a gray sink between two countertops with a faucet in a kitchen, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'long_neck_faucet', 'text': 'a tall long-neck faucet and a sink between two countertops in a kitchen, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'two_tone_layout', 'text': 'a black countertop on one side and a whitish countertop on the other side, and a sink with a faucet in a kitchen, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.1, 'enabled': True},
        ],
    },
    'white_dove_black_feather': {
        'name': 'White dove and black feather',
        'target_prompt': 'A realistic photo of a white dove and a black feather, sharp focus',
        'source_prompt': 'A realistic photo of a dove and a feather, sharp focus',
        'negative_prompt': 'blurry, low quality, missing object, wrong color, fused objects, text, watermark',
        'primitive_prompts': [
            {'name': 'white_dove', 'text': 'A realistic photo of a pure white dove, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'black_feather', 'text': 'A realistic photo of a single black feather, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'two_object_binding', 'text': 'A white dove beside a separate black feather, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.1, 'enabled': True},
        ],
    },
    'marble_fox_beach_snow': {
        'name': 'Marble fox in green rain boots at snowy beach',
        'target_prompt': 'A marble fox wearing green rain boots at the beach while it is snowing, realistic photo, sharp focus',
        'source_prompt': 'A fox standing at the beach, realistic photo, sharp focus',
        'negative_prompt': 'blurry, low quality, deformed fox, extra limbs, missing boots, wrong color, indoor scene, text, watermark',
        'primitive_prompts': [
            {'name': 'marble_fox', 'text': 'A fox made of white marble standing outdoors, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'green_rain_boots', 'text': 'A fox wearing four green rain boots, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'beach_scene', 'text': 'A fox standing on a sandy beach beside the ocean, realistic photo, sharp focus', 'role': 'primitive', 'weight': 0.9, 'enabled': True},
            {'name': 'snowing_weather', 'text': 'A fox outdoors while snow is visibly falling, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'full_binding', 'text': 'A marble fox wearing green rain boots on a beach during falling snow, realistic photo, sharp focus', 'role': 'primitive', 'weight': 1.1, 'enabled': True},
        ],
    },
    'purple_elephant_green_microwave': {
        'name': 'Purple elephant and green microwave',
        'target_prompt': 'A photo of a purple elephant and a green microwave',
        'source_prompt': 'A photo of an elephant and a microwave',
        'negative_prompt': 'blurry, low quality, missing object, wrong color binding, fused objects, text, watermark',
        'primitive_prompts': [
            {'name': 'purple_elephant', 'text': 'A photo of a purple elephant', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'green_microwave', 'text': 'A photo of a green microwave oven', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'two_objects', 'text': 'A photo containing an elephant and a separate microwave oven', 'role': 'primitive', 'weight': 0.9, 'enabled': True},
            {'name': 'full_binding', 'text': 'A photo of a purple elephant beside a separate green microwave oven', 'role': 'primitive', 'weight': 1.1, 'enabled': True},
        ],
    },
    'blue_bench_red_ball': {
        'name': 'Blue bench and red sports ball',
        'target_prompt': 'A photo of a blue bench and a red sports ball',
        'source_prompt': 'A photo of a bench and a sports ball',
        'negative_prompt': 'blurry, low quality, missing object, wrong color binding, fused objects, text, watermark',
        'primitive_prompts': [
            {'name': 'blue_bench', 'text': 'A photo of a blue bench', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'red_sports_ball', 'text': 'A photo of a red sports ball', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
            {'name': 'full_binding', 'text': 'A photo of a blue bench beside a separate red sports ball', 'role': 'primitive', 'weight': 1.1, 'enabled': True},
        ],
    },
}

# Copy, fill, and add this under a new key in PROMPT_DECOMPOSITIONS if needed.
BLANK_DECOMPOSITION_TEMPLATE = {
    'name': '',
    'target_prompt': '',
    'source_prompt': '',
    'negative_prompt': '',
    'primitive_prompts': [
        {'name': '', 'text': '', 'role': 'primitive', 'weight': 1.0, 'enabled': True},
        # Add more complete-sentence primitives here.
    ],
}

PROMPT_CRITERIA = {
    'kitchen_scene': ['black countertop', 'whitish countertop', 'gray sink', 'tall long-neck faucet', 'correct two-sided layout'],
    'white_dove_black_feather': ['white dove', 'separate black feather', 'correct color binding'],
    'marble_fox_beach_snow': ['marble fox material', 'green rain boots', 'beach', 'visible falling snow'],
    'purple_elephant_green_microwave': ['purple elephant', 'separate green microwave', 'correct color binding'],
    'blue_bench_red_ball': ['blue bench', 'separate red sports ball', 'correct color binding'],
}


## Environment, schema validation, and model loading

The next cells find the repository root, validate every decomposition against the repository's `PrimitiveFlowSet`, and load SD3 once. A Hugging Face token with access to the gated SD3 Medium checkpoint must be available as `HF_TOKEN` or `HUGGINGFACE_TOKEN` when generation is enabled.


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import itertools
import json
import math
import re
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
from PIL import Image

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src' / 'aim_flow').is_dir() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not find the aim-flow repository root from the current directory.')

REPO_DIR = find_repo_root(Path.cwd())
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from aim_flow.config import load_config
from aim_flow.prompt_schema import PrimitiveFlowSet
from aim_flow.sampler import AIMFlowSampler
from aim_flow.sd3_backend import SD3Backend

print('Repository:', REPO_DIR)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
missing = sorted(set(ACTIVE_PROMPT_KEYS + SWEEP_PROMPT_KEYS + GRID_PROMPT_KEYS + PRIMITIVE_GALLERY_PROMPT_KEYS + [PRESSURE_PROMPT_KEY]) - set(PROMPT_DECOMPOSITIONS))
if missing:
    raise KeyError(f'Unknown prompt keys: {missing}')

FLOW_SETS = {key: PrimitiveFlowSet.from_dict(value) for key, value in PROMPT_DECOMPOSITIONS.items()}
pressure_names = [primitive.name for primitive in FLOW_SETS[PRESSURE_PROMPT_KEY].primitive_prompts]
if PRESSURE_PRIMITIVE_NAME not in pressure_names:
    raise KeyError(f'Unknown pressure primitive {PRESSURE_PRIMITIVE_NAME!r}; available: {pressure_names}')

lines = ['| key | source | enabled primitives | manual success criteria |', '|---|---|---:|---|']
for key in ALL_PROMPT_KEYS:
    flow = FLOW_SETS[key]
    criteria = '; '.join(PROMPT_CRITERIA[key])
    lines.append(f'| `{key}` | {flow.source_prompt} | {len(flow.get_enabled_primitives())} | {criteria} |')
display(Markdown('## Validated decompositions\n\n' + '\n'.join(lines)))


In [ ]:
BASE_CONFIG = load_config(REPO_DIR / CONFIG_PATH)
BASE_CONFIG.model.model_id = MODEL_ID
BASE_CONFIG.model.dtype = DTYPE
BASE_CONFIG.model.load_t5_text_encoder = LOAD_T5_TEXT_ENCODER
BASE_CONFIG.model.enable_model_cpu_offload = ENABLE_MODEL_CPU_OFFLOAD
BASE_CONFIG.model.enable_vae_slicing = ENABLE_VAE_SLICING
BASE_CONFIG.sampler.height = HEIGHT
BASE_CONFIG.sampler.width = WIDTH
BASE_CONFIG.sampler.num_inference_steps = NUM_INFERENCE_STEPS
BASE_CONFIG.sampler.guidance_scale = GUIDANCE_SCALE
BASE_CONFIG.sampler.seed = SEEDS[0]

for knob, value in BASE_SPFC_KNOBS.items():
    if not hasattr(BASE_CONFIG.primitive_flow, knob):
        raise KeyError(f'Unknown PrimitiveFlowConfig knob: {knob}')
    setattr(BASE_CONFIG.primitive_flow, knob, copy.deepcopy(value))

if not BASE_CONFIG.primitive_flow.enabled or BASE_CONFIG.primitive_flow.mode != 'sparse_primitive_flow':
    raise ValueError('This notebook directly runs sparse primitive flow; keep enabled=True and mode=sparse_primitive_flow.')
if not BASE_CONFIG.primitive_flow.include_target_flow or not BASE_CONFIG.primitive_flow.target_reference:
    raise ValueError('This notebook locks the target condition and target reference to True; see the knob guide.')
if BASE_CONFIG.primitive_flow.min_gate > BASE_CONFIG.primitive_flow.max_gate:
    raise ValueError('min_gate must be <= max_gate.')
if not SEEDS:
    raise ValueError('SEEDS must contain at least one integer.')

BACKEND = None
if RUN_GENERATION:
    if not torch.cuda.is_available():
        raise RuntimeError('RUN_GENERATION=True but CUDA is unavailable.')
    BACKEND = SD3Backend(BASE_CONFIG).load()
    print('Loaded:', MODEL_ID)
else:
    print('Dry planning mode: model not loaded. Set RUN_GENERATION=True in the top cell to generate images.')

planned_core = len(ACTIVE_PROMPT_KEYS) * len(SEEDS) * len(ACTIVE_CORE_ABLATIONS)
planned_sweep = len(SWEEP_PROMPT_KEYS) * len(SEEDS) * len(ONE_FACTOR_SWEEPS[ACTIVE_SWEEP_KNOB])
print(f'Core SPFC runs planned: {planned_core} (+ {len(ACTIVE_PROMPT_KEYS) * len(SEEDS)} base images)')
print(f'One-factor sweep runs planned: {planned_sweep}')


## Reproducible runner and cache

Each output directory contains the image and full sampler metadata. A hash of the prompt decomposition plus every effective setting is appended to the directory name, preventing an old image from being mistaken for a new configuration. Rerunning a cell reuses matching cached outputs unless `OVERWRITE_EXISTING=True`.


In [ ]:
OUTPUT_DIR = (REPO_DIR / OUTPUT_ROOT).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_REGISTRY = {}

def slugify(value: object) -> str:
    text = re.sub(r'[^a-zA-Z0-9._-]+', '-', str(value)).strip('-').lower()
    return text or 'run'

def apply_spfc_overrides(config, overrides: dict):
    cfg = copy.deepcopy(config)
    for knob, value in overrides.items():
        if not hasattr(cfg.primitive_flow, knob):
            raise KeyError(f'Unknown PrimitiveFlowConfig knob: {knob}')
        setattr(cfg.primitive_flow, knob, copy.deepcopy(value))
    pf = cfg.primitive_flow
    if not pf.enabled or pf.mode != 'sparse_primitive_flow':
        raise ValueError('Keep enabled=True and mode=sparse_primitive_flow in this direct SPFC runner.')
    if not pf.include_target_flow or not pf.target_reference:
        raise ValueError('Target-free/reference-free runs are not valid in the current SPFC sampler.')
    if pf.min_gate > pf.max_gate:
        raise ValueError(f'min_gate ({pf.min_gate}) must be <= max_gate ({pf.max_gate}).')
    if pf.vfa_temperature <= 0 or pf.velocity_clip_ratio < 0 or pf.steering_strength < 0 or pf.ltp_radius_ratio < 0:
        raise ValueError('Temperature must be > 0 and clip/steering/LTP ratios must be non-negative.')
    if pf.ltp_mode == 'off':
        pf.ltp_enabled = False
    return cfg

def settings_payload(flow: PrimitiveFlowSet, config, variant: str) -> dict:
    return {
        'variant': variant,
        'prompt': flow.to_dict(),
        'model': config.to_dict()['model'],
        'sampler': config.to_dict()['sampler'],
        'primitive_flow': config.to_dict()['primitive_flow'],
    }

def remember(record: dict) -> dict:
    RUN_REGISTRY[(record['prompt_key'], record['variant'], record['seed'], record['fingerprint'])] = record
    return record

def run_base_target(prompt_key: str, seed: int) -> dict:
    flow = FLOW_SETS[prompt_key]
    cfg = copy.deepcopy(BASE_CONFIG)
    cfg.sampler.seed = int(seed)
    payload = {'variant': 'base_target', 'prompt': flow.to_dict(), 'model': cfg.to_dict()['model'], 'sampler': cfg.to_dict()['sampler']}
    fingerprint = hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]
    run_dir = OUTPUT_DIR / prompt_key / f'seed_{seed}' / f'base_target-{fingerprint}'
    image_path = run_dir / 'image.png'
    metadata_path = run_dir / 'metadata.json'
    record = {'prompt_key': prompt_key, 'variant': 'base_target', 'display_label': 'Base target', 'seed': seed, 'fingerprint': fingerprint, 'image_path': image_path, 'metadata_path': metadata_path, 'overrides': {}, 'status': 'planned'}
    if image_path.exists() and metadata_path.exists() and not OVERWRITE_EXISTING:
        record['metadata'] = json.loads(metadata_path.read_text())
        record['status'] = 'cached'
        return remember(record)
    if not RUN_GENERATION:
        return remember(record)
    run_dir.mkdir(parents=True, exist_ok=True)
    image = BACKEND.generate_base(flow.target_prompt, flow.negative_prompt, seed, NUM_INFERENCE_STEPS, GUIDANCE_SCALE, HEIGHT, WIDTH)
    BACKEND.save_image(image, image_path)
    metadata = {'method': 'base_target', **payload}
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True))
    record.update({'metadata': metadata, 'status': 'generated'})
    return remember(record)

def run_spfc(prompt_key: str, variant: str, overrides: dict, seed: int, flow_override: PrimitiveFlowSet | None = None) -> dict:
    flow = flow_override or FLOW_SETS[prompt_key]
    cfg = apply_spfc_overrides(BASE_CONFIG, overrides)
    cfg.sampler.seed = int(seed)
    payload = settings_payload(flow, cfg, variant)
    fingerprint = hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]
    run_dir = OUTPUT_DIR / prompt_key / f'seed_{seed}' / f'{slugify(variant)}-{fingerprint}'
    image_path = run_dir / 'image.png'
    metadata_path = run_dir / 'metadata.json'
    record = {'prompt_key': prompt_key, 'variant': variant, 'display_label': variant.replace('_', ' '), 'seed': seed, 'fingerprint': fingerprint, 'image_path': image_path, 'metadata_path': metadata_path, 'overrides': copy.deepcopy(overrides), 'status': 'planned'}
    if image_path.exists() and metadata_path.exists() and not OVERWRITE_EXISTING:
        record['metadata'] = json.loads(metadata_path.read_text())
        record['status'] = 'cached'
        return remember(record)
    if not RUN_GENERATION:
        return remember(record)
    run_dir.mkdir(parents=True, exist_ok=True)
    sampler = AIMFlowSampler(BACKEND, cfg)
    image, metadata = sampler.generate_sparse_primitive_flow(flow, mode='primitive_flow_sparse')
    metadata['ablation_variant'] = variant
    metadata['ablation_overrides'] = overrides
    metadata['settings_fingerprint'] = fingerprint
    BACKEND.save_image(image, image_path)
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True))
    record.update({'metadata': metadata, 'status': 'generated'})
    return remember(record)

def run_variant_over_prompts(variant: str, overrides: dict, prompt_keys=None, seeds=None) -> list[dict]:
    prompt_keys = prompt_keys or ACTIVE_PROMPT_KEYS
    seeds = seeds or SEEDS
    records = []
    for seed in seeds:
        for prompt_key in prompt_keys:
            print(f'[{variant}] seed={seed} prompt={prompt_key}')
            records.append(run_spfc(prompt_key, variant, overrides, seed))
    return records


In [ ]:
def run_standalone_condition(
    prompt_key: str,
    condition_key: str,
    condition_name: str,
    condition_role: str,
    condition_text: str,
    condition_order: int,
    seed: int,
) -> dict:
    flow = FLOW_SETS[prompt_key]
    cfg = copy.deepcopy(BASE_CONFIG)
    cfg.sampler.seed = int(seed)
    variant = f'standalone::{condition_key}'
    payload = {
        'method': 'standalone_condition', 'prompt_key': prompt_key, 'variant': variant,
        'condition_name': condition_name, 'condition_role': condition_role, 'condition_text': condition_text,
        'negative_prompt': flow.negative_prompt, 'model': cfg.to_dict()['model'], 'sampler': cfg.to_dict()['sampler'],
    }
    fingerprint = hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]
    run_dir = OUTPUT_DIR / prompt_key / f'seed_{seed}' / 'standalone_conditions' / f'{slugify(condition_key)}-{fingerprint}'
    image_path = run_dir / 'image.png'
    metadata_path = run_dir / 'metadata.json'
    record = {
        'prompt_key': prompt_key, 'variant': variant, 'display_label': condition_name, 'seed': int(seed),
        'fingerprint': fingerprint, 'image_path': image_path, 'metadata_path': metadata_path,
        'overrides': {}, 'status': 'planned', 'condition_key': condition_key,
        'condition_name': condition_name, 'condition_role': condition_role,
        'condition_text': condition_text, 'condition_order': int(condition_order),
    }
    if image_path.exists() and metadata_path.exists() and not OVERWRITE_EXISTING:
        record['metadata'] = json.loads(metadata_path.read_text())
        record['status'] = 'cached'
        return remember(record)
    if not RUN_GENERATION:
        return remember(record)
    run_dir.mkdir(parents=True, exist_ok=True)
    image = BACKEND.generate_base(
        condition_text, flow.negative_prompt, seed, NUM_INFERENCE_STEPS, GUIDANCE_SCALE, HEIGHT, WIDTH
    )
    BACKEND.save_image(image, image_path)
    metadata_path.write_text(json.dumps(payload, indent=2, sort_keys=True))
    record.update({'metadata': payload, 'status': 'generated'})
    return remember(record)

def run_standalone_primitive_gallery(prompt_keys=None, seeds=None) -> list[dict]:
    prompt_keys = prompt_keys or PRIMITIVE_GALLERY_PROMPT_KEYS
    seeds = seeds or SEEDS
    records = []
    for seed in seeds:
        for prompt_key in prompt_keys:
            flow = FLOW_SETS[prompt_key]
            primitives = flow.get_enabled_primitives()
            if PRIMITIVE_GALLERY_MAX_PRIMITIVES is not None:
                primitives = primitives[:max(0, int(PRIMITIVE_GALLERY_MAX_PRIMITIVES))]
            order = 0
            if PRIMITIVE_GALLERY_INCLUDE_SOURCE:
                print(f'[standalone source] seed={seed} prompt={prompt_key}')
                records.append(run_standalone_condition(prompt_key, 'source', 'source', 'source', flow.source_prompt, order, seed))
                order += 1
            for index, primitive in enumerate(primitives):
                name = primitive.name or f'primitive_{index}'
                print(f'[standalone primitive] seed={seed} prompt={prompt_key} primitive={name}')
                records.append(run_standalone_condition(prompt_key, f'primitive_{index}_{name}', name, 'primitive', primitive.text, order, seed))
                order += 1
            if PRIMITIVE_GALLERY_INCLUDE_TARGET:
                print(f'[standalone target] seed={seed} prompt={prompt_key}')
                records.append(run_standalone_condition(prompt_key, 'target', 'target', 'target', flow.target_prompt, order, seed))
    return records

def show_standalone_primitive_gallery(records: list[dict], prompt_keys=None, seed=None, save_name='standalone_primitive_gallery'):
    prompt_keys = prompt_keys or PRIMITIVE_GALLERY_PROMPT_KEYS
    seed = PRESSURE_SEED if seed is None else seed
    grouped = {key: sorted([r for r in records if r['prompt_key'] == key and r['seed'] == seed], key=lambda r: r['condition_order']) for key in prompt_keys}
    max_columns = max([len(row) for row in grouped.values()] or [1])
    fig, axes = plt.subplots(len(prompt_keys), max_columns, figsize=(4.2 * max_columns, 4.8 * len(prompt_keys)), squeeze=False)
    prompt_lines = []
    for row_index, prompt_key in enumerate(prompt_keys):
        row_records = grouped[prompt_key]
        for column_index in range(max_columns):
            ax = axes[row_index][column_index]
            if column_index < len(row_records):
                record = row_records[column_index]
                if Path(record['image_path']).exists():
                    with Image.open(record['image_path']) as image:
                        ax.imshow(image.convert('RGB'))
                else:
                    ax.text(0.5, 0.5, record['status'], ha='center', va='center')
                short_text = record['condition_text'] if len(record['condition_text']) <= 70 else record['condition_text'][:67] + '...'
                ax.set_title(f"{record['condition_role']}: {record['condition_name']}\n{short_text}", fontsize=9)
                prompt_lines.append(f"| `{prompt_key}` | {record['condition_role']} | `{record['condition_name']}` | {record['condition_text']} |")
            else:
                ax.axis('off')
            ax.set_xticks([]); ax.set_yticks([])
            if column_index == 0:
                ax.set_ylabel(prompt_key.replace('_', ' '), fontsize=10)
    fig.suptitle(f'Standalone source / primitive / target generations — seed {seed}', fontsize=15)
    fig.tight_layout()
    if save_name:
        grid_dir = OUTPUT_DIR / 'grids'
        grid_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(grid_dir / f'{slugify(save_name)}_seed_{seed}.png', dpi=150, bbox_inches='tight')
    plt.show()
    if prompt_lines:
        display(Markdown('\n'.join(['| prompt | role | condition | exact text |', '|---|---|---|---|', *prompt_lines])))

def flow_with_primitive_weight(prompt_key: str, primitive_name: str, weight: float) -> PrimitiveFlowSet:
    data = copy.deepcopy(FLOW_SETS[prompt_key].to_dict())
    matches = [item for item in data['primitive_prompts'] if item['name'] == primitive_name]
    if len(matches) != 1:
        raise KeyError(f'Expected one primitive named {primitive_name!r}; found {len(matches)}.')
    matches[0]['weight'] = float(weight)
    return PrimitiveFlowSet.from_dict(data)

def flow_with_primitive_duplicates(prompt_key: str, primitive_name: str, total_copies: int) -> PrimitiveFlowSet:
    if total_copies < 1:
        raise ValueError('total_copies must be at least 1.')
    data = copy.deepcopy(FLOW_SETS[prompt_key].to_dict())
    primitives = data['primitive_prompts']
    indices = [index for index, item in enumerate(primitives) if item['name'] == primitive_name]
    if len(indices) != 1:
        raise KeyError(f'Expected one primitive named {primitive_name!r}; found {len(indices)}.')
    source_index = indices[0]
    original = copy.deepcopy(primitives[source_index])
    duplicates = []
    for copy_index in range(1, total_copies):
        duplicate = copy.deepcopy(original)
        duplicate['name'] = f'{primitive_name}_duplicate_{copy_index}'
        duplicates.append(duplicate)
    data['primitive_prompts'] = primitives[:source_index + 1] + duplicates + primitives[source_index + 1:]
    return PrimitiveFlowSet.from_dict(data)


In [ ]:
def show_comparison_matrix(records: list[dict], prompt_order: list[str], variant_order: list[str], seed=DISPLAY_SEED, save_name=None):
    lookup = {(r['prompt_key'], r['variant'], r['seed']): r for r in records}
    rows, cols = len(prompt_order), len(variant_order)
    fig, axes = plt.subplots(rows, cols, figsize=(4.0 * cols, 4.4 * rows), squeeze=False)
    for row, prompt_key in enumerate(prompt_order):
        for col, variant in enumerate(variant_order):
            ax = axes[row][col]
            record = lookup.get((prompt_key, variant, seed))
            if record and Path(record['image_path']).exists():
                with Image.open(record['image_path']) as image:
                    ax.imshow(image.convert('RGB'))
            else:
                status = record['status'] if record else 'not planned'
                ax.text(0.5, 0.5, status, ha='center', va='center', fontsize=12)
            if row == 0:
                ax.set_title(variant.replace('_', ' '), fontsize=11)
            if col == 0:
                ax.set_ylabel(prompt_key.replace('_', ' '), fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'SPFC comparison — seed {seed}', fontsize=15)
    fig.tight_layout()
    if save_name:
        grid_dir = OUTPUT_DIR / 'grids'
        grid_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(grid_dir / f'{slugify(save_name)}_seed_{seed}.png', dpi=150, bbox_inches='tight')
    plt.show()

def load_record_metadata(record: dict) -> dict | None:
    if record.get('metadata') is not None:
        return record['metadata']
    path = Path(record['metadata_path'])
    return json.loads(path.read_text()) if path.exists() else None

def safe_mean(values):
    values = [float(v) for v in values if v is not None and math.isfinite(float(v))]
    return statistics.fmean(values) if values else None

def summarize_diagnostics(record: dict) -> dict | None:
    metadata = load_record_metadata(record)
    if not metadata or 'debug_steps' not in metadata:
        return None
    steps = [step for step in metadata['debug_steps'] if step.get('do_aggregate') and step.get('vfa')]
    if not steps:
        return {'prompt_key': record['prompt_key'], 'variant': record['variant'], 'seed': record['seed'], 'aggregation_steps': 0}
    raw = [step['vfa'].get('raw_correction_norm') for step in steps]
    clipped = [step['vfa'].get('clipped_correction_norm') for step in steps]
    steered = [step['vfa'].get('steered_correction_norm') for step in steps]
    clamp_hits = [float(c) < float(r) * (1 - 1e-6) for r, c in zip(raw, clipped) if r and c is not None]
    target_weights, entropies, consensus_means, target_gate_means = [], [], [], []
    ltp_retentions = []
    for step in steps:
        vfa = step['vfa']
        weights = vfa.get('softmax_weights') or []
        target_index = int(vfa.get('target_index', step.get('target_index', len(weights) - 1)))
        if weights:
            target_weights.append(weights[target_index])
            if len(weights) > 1:
                entropy = -sum(w * math.log(max(w, 1e-12)) for w in weights) / math.log(len(weights))
                entropies.append(entropy)
        consensus_means.append(safe_mean(vfa.get('consensus_gates') or []))
        target_gate_means.append(safe_mean(vfa.get('target_consistency_gates') or []))
        ltp = step.get('ltp_debug') or {}
        before = ltp.get('candidate_offset_norm', ltp.get('candidate_correction_norm'))
        after = ltp.get('projected_offset_norm', ltp.get('projected_correction_norm'))
        if before is not None and float(before) > 0 and after is not None:
            ltp_retentions.append(float(after) / float(before))
    return {
        'prompt_key': record['prompt_key'], 'variant': record['variant'], 'seed': record['seed'],
        'aggregation_steps': len(steps),
        'mean_raw_correction': safe_mean(raw),
        'mean_clipped_correction': safe_mean(clipped),
        'mean_steered_correction': safe_mean(steered),
        'velocity_clamp_hit_rate': safe_mean(clamp_hits),
        'mean_ltp_retention': safe_mean(ltp_retentions),
        'mean_target_softmax_weight': safe_mean(target_weights),
        'mean_weight_entropy': safe_mean(entropies),
        'mean_consensus_gate': safe_mean(consensus_means),
        'mean_target_gate': safe_mean(target_gate_means),
    }

def show_diagnostic_table(records: list[dict]):
    summaries = [summary for record in records if (summary := summarize_diagnostics(record)) is not None]
    if not summaries:
        print('No generated SPFC metadata is available yet.')
        return []
    columns = ['prompt_key', 'variant', 'seed', 'aggregation_steps', 'velocity_clamp_hit_rate', 'mean_ltp_retention', 'mean_target_softmax_weight', 'mean_weight_entropy']
    lines = ['| ' + ' | '.join(columns) + ' |', '|' + '|'.join(['---'] * len(columns)) + '|']
    for summary in summaries:
        values = []
        for column in columns:
            value = summary.get(column)
            values.append(f'{value:.3f}' if isinstance(value, float) else str(value))
        lines.append('| ' + ' | '.join(values) + ' |')
    display(Markdown('\n'.join(lines)))
    return summaries

def plot_step_diagnostics(record: dict):
    metadata = load_record_metadata(record)
    if not metadata:
        print('No metadata for this record.')
        return
    steps = [step for step in metadata.get('debug_steps', []) if step.get('do_aggregate') and step.get('vfa')]
    if not steps:
        print('This run has no aggregation-step diagnostics.')
        return
    x = [step['step_index'] for step in steps]
    raw = [step['vfa']['raw_correction_norm'] for step in steps]
    clipped = [step['vfa']['clipped_correction_norm'] for step in steps]
    steered = [step['vfa']['steered_correction_norm'] for step in steps]
    target_w = [step['vfa']['softmax_weights'][step['vfa']['target_index']] for step in steps]
    consensus = [safe_mean(step['vfa']['consensus_gates']) for step in steps]
    target_gate = [safe_mean(step['vfa']['target_consistency_gates']) for step in steps]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(x, raw, 'o-', label='raw correction')
    axes[0].plot(x, clipped, 'o-', label='after velocity clip')
    axes[0].plot(x, steered, 'o-', label='after steering')
    axes[0].set_title('Correction norms'); axes[0].set_xlabel('step'); axes[0].legend()
    axes[1].plot(x, target_w, 'o-', color='black')
    axes[1].set_title('Target softmax weight'); axes[1].set_xlabel('step'); axes[1].set_ylim(0, 1)
    axes[2].plot(x, consensus, 'o-', label='mean consensus gate')
    axes[2].plot(x, target_gate, 'o-', label='mean target gate')
    axes[2].set_title('Gate means'); axes[2].set_xlabel('step'); axes[2].set_ylim(0, 1); axes[2].legend()
    fig.suptitle(f"{record['prompt_key']} — {record['variant']} — seed {record['seed']}")
    fig.tight_layout(); plt.show()

def show_two_factor_grid(records, prompt_key, x_knob, x_values, y_knob, y_values, seed=DISPLAY_SEED):
    lookup = {(r['overrides'].get(x_knob), r['overrides'].get(y_knob), r['seed']): r for r in records if r['prompt_key'] == prompt_key}
    fig, axes = plt.subplots(len(y_values), len(x_values), figsize=(4 * len(x_values), 4.3 * len(y_values)), squeeze=False)
    for row, y_value in enumerate(y_values):
        for col, x_value in enumerate(x_values):
            ax = axes[row][col]
            record = lookup.get((x_value, y_value, seed))
            if record and Path(record['image_path']).exists():
                with Image.open(record['image_path']) as image:
                    ax.imshow(image.convert('RGB'))
            else:
                ax.text(0.5, 0.5, record['status'] if record else 'missing', ha='center', va='center')
            if row == 0: ax.set_title(f'{x_knob}={x_value}')
            if col == 0: ax.set_ylabel(f'{y_knob}={y_value}')
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'{prompt_key} — seed {seed}')
    fig.tight_layout(); plt.show()


## Stage 0 — establish the target-only baseline

Generate the ordinary SD3 target image for every prompt and seed. This is the baseline SPFC must beat. The `steering_zero_control` and `no_aggregation_control` variants in Stage 1 are implementation controls: both should remain on the target trajectory even though they pass through the custom sampler.


In [ ]:
baseline_records = []
for seed in SEEDS:
    for prompt_key in ACTIVE_PROMPT_KEYS:
        print(f'[base_target] seed={seed} prompt={prompt_key}')
        baseline_records.append(run_base_target(prompt_key, seed))

show_comparison_matrix(baseline_records, ACTIVE_PROMPT_KEYS, ['base_target'], save_name='stage_0_base_targets')


## Stage 0.5 — render every condition by itself

Enable `RUN_STANDALONE_PRIMITIVE_GALLERY` in the first cell to generate the source prompt, every enabled primitive prompt, and the full target independently from the **same seed and sampler settings**. This answers the first prerequisite question: can the frozen model render the intended concept when that primitive has the entire conditioning channel to itself?

A failed standalone primitive usually indicates a model-capacity or wording problem; VFA cannot reliably recover a concept whose own conditional flow does not express it. A successful standalone primitive does not prove that it contributes strongly inside SPFC, because SPFC later gates, normalizes, clips, and projects its velocity.


In [ ]:
standalone_primitive_records = []
if RUN_STANDALONE_PRIMITIVE_GALLERY:
    standalone_primitive_records = run_standalone_primitive_gallery(PRIMITIVE_GALLERY_PROMPT_KEYS, SEEDS)
    for gallery_seed in SEEDS:
        show_standalone_primitive_gallery(standalone_primitive_records, PRIMITIVE_GALLERY_PROMPT_KEYS, gallery_seed)
else:
    print('Standalone primitive gallery disabled. Set RUN_STANDALONE_PRIMITIVE_GALLERY=True in the first cell.')


## Stage 1 — component ablations

Run the selected named variants across the same five prompts. The safest causal reading comes from comparing each column directly with `full_spfc` while changing exactly one mechanism. Add `unbounded_correction` only after inspecting `no_velocity_clip` and `no_ltp` separately; removing both safeguards at once can destabilize sampling.


In [ ]:
unknown = sorted(set(ACTIVE_CORE_ABLATIONS) - set(CORE_ABLATIONS))
if unknown:
    raise KeyError(f'Unknown core ablations: {unknown}')

core_records = list(baseline_records)
for variant in ACTIVE_CORE_ABLATIONS:
    core_records.extend(run_variant_over_prompts(variant, CORE_ABLATIONS[variant]))

core_columns = ['base_target', *ACTIVE_CORE_ABLATIONS]
show_comparison_matrix(core_records, ACTIVE_PROMPT_KEYS, core_columns, save_name='stage_1_core_ablations')
core_diagnostics = show_diagnostic_table(core_records)


## Why does SPFC appear to pursue a primitive aggressively?

The model is not explicitly checking whether a primitive was fulfilled. What looks like intent is repeated numerical influence on the latent. Because the VFA weights sum to one, its target-relative correction can be written exactly as:

`delta_raw = v_raw - v_target = sum_i alpha_i * (v_i - v_target)`

A primitive is influential only when it receives allocation `alpha_i` **and** its velocity differs meaningfully from the target velocity in a useful direction. A high softmax weight alone is therefore an allocation proxy, not a complete measure of causal contribution.

### Working hypotheses

| Hypothesis | Why it could look aggressive | How this notebook tests it |
|---|---|---|
| **Standalone semantic capacity** | An explicit, visually grounded sentence may make the frozen model produce a much clearer conditional direction than the crowded full prompt. | Stage 0.5 renders every primitive alone. |
| **VFA allocation** | Base weight, consensus, target agreement, and low temperature can give one primitive a larger `alpha_i`. | The arbitration audit below plots raw scores, gates, and softmax weights over time; the weight sweep changes one primitive prior. |
| **Residual leverage** | Even moderate `alpha_i` can matter if `v_i - v_target` is large. Conversely, a target-like primitive can receive high weight but add almost no correction. | Compare standalone distinctness and correction norms. Exact per-condition residual norms are not currently logged, so do not interpret `alpha_i` alone as effect size. |
| **Condition-count or coalition bias** | Rephrasing or duplicating the same concept gives it multiple softmax slots. The copies also agree with each other, which can improve consensus. Their collective mass may steer the mixture toward that concept. | The optional duplication test holds the text fixed and varies only the number of copies. |
| **Temporal repetition** | A primitive evaluated at many aggregation steps repeatedly bends the path. Early steps have disproportionate leverage over layout and object identity. | Compare dense, early-only, late-only, and final-only schedules. |
| **Global gain and weak safeguards** | Larger steering strength, a loose velocity clip, or a loose/disabled LTP trust region lets the composed correction travel farther from the target path. | Existing clip, steering, and LTP sweeps expose this interaction. |
| **CFG amplification** | Every condition is CFG-guided before VFA, so a higher guidance scale can make primitive-conditioned velocities more assertive as well as change their agreement geometry. | Treat CFG as a later outer-loop sweep; keep it fixed during VFA attribution. |

The strongest specific hypothesis is **coalition plus repetition**: several overlapping primitives can collectively occupy substantial softmax mass, raise one another's consensus, and apply a similar semantic correction at every selected step. With five primitive slots plus source and target, the primitives can hold most of the collective mixture even when no individual primitive beats the target.

The velocity clip and LTP bounds are **local per-step constraints, not a global guarantee of closeness to the original target-only sample**. LTP projects around the target step computed from the *current, already steered latent*. Small permitted offsets can therefore accumulate over a dense schedule. This makes dense all-step aggregation a particularly strong explanation for apparently aggressive primitive fulfillment.


In [ ]:
def show_primitive_arbitration(record: dict):
    metadata = load_record_metadata(record)
    if not metadata:
        print('No metadata is available for this run.')
        return
    steps = [step for step in metadata.get('debug_steps', []) if step.get('do_aggregate') and step.get('vfa')]
    if not steps:
        print('The selected run has no VFA aggregation steps.')
        return
    first_vfa = steps[0]['vfa']
    names = first_vfa['condition_names']
    roles = first_vfa['condition_roles']
    step_indices = [step['step_index'] for step in steps]
    weights = [[step['vfa']['softmax_weights'][index] for step in steps] for index in range(len(names))]
    raw_scores = [[step['vfa']['raw_scores'][index] for step in steps] for index in range(len(names))]
    consensus = [[step['vfa']['consensus_gates'][index] for step in steps] for index in range(len(names))]
    target_gates = [[step['vfa']['target_consistency_gates'][index] for step in steps] for index in range(len(names))]

    lines = ['| condition | role | base weight | mean raw score | mean softmax allocation | mean consensus | mean target gate |', '|---|---|---:|---:|---:|---:|---:|']
    for index, (name, role) in enumerate(zip(names, roles)):
        lines.append(
            f'| `{name}` | {role} | {first_vfa["condition_base_weights"][index]:.3f} | '
            f'{safe_mean(raw_scores[index]):.3f} | {safe_mean(weights[index]):.3f} | '
            f'{safe_mean(consensus[index]):.3f} | {safe_mean(target_gates[index]):.3f} |'
        )
    display(Markdown('\n'.join(lines)))

    fig, axes = plt.subplots(2, 2, figsize=(17, 11))
    panels = [
        (weights, 'Softmax allocation alpha', 0.0, 1.0),
        (raw_scores, 'Raw scores before softmax', None, None),
        (target_gates, 'Target-consistency gates', 0.0, 1.0),
    ]
    for ax, (matrix, title, vmin, vmax) in zip(axes.flat[:3], panels):
        image = ax.imshow(matrix, aspect='auto', interpolation='nearest', vmin=vmin, vmax=vmax, cmap='viridis')
        ax.set_title(title); ax.set_yticks(range(len(names)), names); ax.set_xticks(range(len(step_indices)), step_indices, rotation=90)
        ax.set_xlabel('aggregation step'); fig.colorbar(image, ax=ax, fraction=0.046)
    pairwise = first_vfa['pairwise_cosine_matrix']
    image = axes[1, 1].imshow(pairwise, vmin=-1.0, vmax=1.0, cmap='coolwarm')
    axes[1, 1].set_title(f'Pairwise velocity cosine at step {step_indices[0]}')
    axes[1, 1].set_xticks(range(len(names)), names, rotation=90); axes[1, 1].set_yticks(range(len(names)), names)
    fig.colorbar(image, ax=axes[1, 1], fraction=0.046)
    fig.suptitle(f"Primitive arbitration: {record['prompt_key']} — {record['variant']} — seed {record['seed']}")
    fig.tight_layout(); plt.show()

def find_run(records: list[dict], prompt_key: str, variant: str, seed: int) -> dict | None:
    return next((record for record in records if record['prompt_key'] == prompt_key and record['variant'] == variant and record['seed'] == seed), None)


In [ ]:
pressure_record = find_run(core_records, PRESSURE_PROMPT_KEY, 'full_spfc', PRESSURE_SEED)
if pressure_record is None:
    print('No matching full_spfc record. Ensure PRESSURE_PROMPT_KEY and PRESSURE_SEED were included in Stage 1.')
elif not Path(pressure_record['metadata_path']).exists():
    print('The pressure record is planned but not generated yet.')
else:
    show_primitive_arbitration(pressure_record)


### Optional direct pressure tests

The first test changes only the selected primitive's base weight. Remember that weight zero does not remove its softmax slot—`exp(0)` is still positive—so use `enabled=False` in the decomposition for a true removal ablation. The second test duplicates identical primitive text to measure condition-count/coalition bias. Duplication is intentionally artificial and can be memory-intensive; it is a mechanism probe, not a recommended prompt design.


In [ ]:
primitive_weight_records = []
primitive_weight_variants = []
if RUN_PRIMITIVE_WEIGHT_SWEEP:
    for primitive_weight in PRIMITIVE_WEIGHT_VALUES:
        variant = f'primitive_weight::{PRESSURE_PRIMITIVE_NAME}={primitive_weight}'
        primitive_weight_variants.append(variant)
        altered_flow = flow_with_primitive_weight(PRESSURE_PROMPT_KEY, PRESSURE_PRIMITIVE_NAME, primitive_weight)
        primitive_weight_records.append(
            run_spfc(PRESSURE_PROMPT_KEY, variant, {'uniform_condition_weights': False}, PRESSURE_SEED, flow_override=altered_flow)
        )
    show_comparison_matrix(primitive_weight_records, [PRESSURE_PROMPT_KEY], primitive_weight_variants, seed=PRESSURE_SEED, save_name=f'primitive_weight_{PRESSURE_PRIMITIVE_NAME}')
    show_diagnostic_table(primitive_weight_records)
else:
    print('Primitive weight sweep disabled. Set RUN_PRIMITIVE_WEIGHT_SWEEP=True in the first cell.')


In [ ]:
primitive_duplication_records = []
primitive_duplication_variants = []
if RUN_PRIMITIVE_DUPLICATION_TEST:
    for copy_count in PRIMITIVE_DUPLICATION_COUNTS:
        variant = f'primitive_copies::{PRESSURE_PRIMITIVE_NAME}={copy_count}'
        primitive_duplication_variants.append(variant)
        altered_flow = flow_with_primitive_duplicates(PRESSURE_PROMPT_KEY, PRESSURE_PRIMITIVE_NAME, copy_count)
        primitive_count = len(altered_flow.get_enabled_primitives())
        primitive_duplication_records.append(
            run_spfc(PRESSURE_PROMPT_KEY, variant, {'uniform_condition_weights': False, 'max_primitives': primitive_count}, PRESSURE_SEED, flow_override=altered_flow)
        )
    show_comparison_matrix(primitive_duplication_records, [PRESSURE_PROMPT_KEY], primitive_duplication_variants, seed=PRESSURE_SEED, save_name=f'primitive_copies_{PRESSURE_PRIMITIVE_NAME}')
    show_diagnostic_table(primitive_duplication_records)
else:
    print('Primitive duplication test disabled. Set RUN_PRIMITIVE_DUPLICATION_TEST=True in the first cell.')


## Stage 2 — one-factor sweep

Choose one entry from `ONE_FACTOR_SWEEPS` in the top cell. Keep the prompt set small initially. For clamp sweeps, inspect both the images and `velocity_clamp_hit_rate`: if two ratios never activate the clamp, they are operationally the same on that sample. For LTP, a retention near `1.0` means the projection was inactive; a lower value means it removed more of the candidate offset.


In [ ]:
if ACTIVE_SWEEP_KNOB not in ONE_FACTOR_SWEEPS:
    raise KeyError(f'Unknown sweep knob: {ACTIVE_SWEEP_KNOB}')

sweep_records = []
sweep_variants = []
for value in ONE_FACTOR_SWEEPS[ACTIVE_SWEEP_KNOB]:
    variant = f'{ACTIVE_SWEEP_KNOB}={value}'
    sweep_variants.append(variant)
    sweep_records.extend(run_variant_over_prompts(variant, {ACTIVE_SWEEP_KNOB: value}, SWEEP_PROMPT_KEYS))

show_comparison_matrix(sweep_records, SWEEP_PROMPT_KEYS, sweep_variants, save_name=f'stage_2_sweep_{ACTIVE_SWEEP_KNOB}')
sweep_diagnostics = show_diagnostic_table(sweep_records)


## Inspect one run across timesteps

Set the selector below to any generated prompt/variant. The plots reveal whether a visible image change coincides with correction clipping, a change in target weight, or disagreement among conditions.


In [ ]:
DIAGNOSTIC_PROMPT_KEY = SWEEP_PROMPT_KEYS[0]
DIAGNOSTIC_VARIANT = sweep_variants[0]

diagnostic_record = next((r for r in sweep_records if r['prompt_key'] == DIAGNOSTIC_PROMPT_KEY and r['variant'] == DIAGNOSTIC_VARIANT and r['seed'] == DISPLAY_SEED), None)
if diagnostic_record is None:
    print('No matching record. Change DIAGNOSTIC_PROMPT_KEY / DIAGNOSTIC_VARIANT.')
else:
    plot_step_diagnostics(diagnostic_record)


## Stage 3 — optional two-factor interaction grid

Run this only after one-factor sweeps. The default grid tests the interaction implied by the implementation: velocity clipping happens first, steering strength multiplies the clipped correction second, and LTP constrains the resulting latent update last. Enable `RUN_TWO_FACTOR_GRID` in the top cell.


In [ ]:
grid_records = []
if RUN_TWO_FACTOR_GRID:
    for x_value, y_value in itertools.product(GRID_X_VALUES, GRID_Y_VALUES):
        variant = f'{GRID_X_KNOB}={x_value}__{GRID_Y_KNOB}={y_value}'
        overrides = {GRID_X_KNOB: x_value, GRID_Y_KNOB: y_value}
        grid_records.extend(run_variant_over_prompts(variant, overrides, GRID_PROMPT_KEYS))
    for prompt_key in GRID_PROMPT_KEYS:
        show_two_factor_grid(grid_records, prompt_key, GRID_X_KNOB, GRID_X_VALUES, GRID_Y_KNOB, GRID_Y_VALUES)
    grid_diagnostics = show_diagnostic_table(grid_records)
else:
    print('Two-factor grid disabled. Set RUN_TWO_FACTOR_GRID=True in the top cell when ready.')


## Manual evaluation and ranking

Rate images blind to the setting name when possible. Use the prompt-specific checklist printed near the top, then assign every metric from 1 to 5. Do not select a winner from one seed: use these ratings to shortlist settings, add several seeds, and validate again across all five prompts.

Fill `MANUAL_RATINGS` below. Tuple keys are `(prompt_key, variant, seed)`. The ranking reports both individual images and the mean score by variant.


In [ ]:
# EDIT HERE after viewing generated images. All values are 1..5.
MANUAL_RATINGS = {
    # Example (remove the leading # and edit):
    # ('kitchen_scene', 'full_spfc', 13): {
    #     'target_fidelity': 4, 'attribute_binding': 4, 'object_coverage': 5,
    #     'composition': 4, 'visual_quality': 4, 'notes': 'Gray sink is slightly too dark.'
    # },
}

def validate_and_rank_ratings(ratings: dict):
    rows = []
    for (prompt_key, variant, seed), values in ratings.items():
        missing_metrics = [metric for metric in MANUAL_RATING_METRICS if metric not in values]
        if missing_metrics:
            raise ValueError(f'{(prompt_key, variant, seed)} is missing {missing_metrics}')
        scores = [float(values[metric]) for metric in MANUAL_RATING_METRICS]
        if any(score < 1 or score > 5 for score in scores):
            raise ValueError(f'Ratings must be in [1,5]: {(prompt_key, variant, seed)}')
        rows.append({
            'prompt_key': prompt_key, 'variant': variant, 'seed': int(seed),
            **{metric: float(values[metric]) for metric in MANUAL_RATING_METRICS},
            'mean_score': statistics.fmean(scores), 'notes': values.get('notes', ''),
        })
    rows.sort(key=lambda row: row['mean_score'], reverse=True)
    grouped = {}
    for row in rows:
        grouped.setdefault(row['variant'], []).append(row['mean_score'])
    ranking = sorted(((variant, statistics.fmean(scores), len(scores)) for variant, scores in grouped.items()), key=lambda item: item[1], reverse=True)
    return rows, ranking

rating_rows, variant_ranking = validate_and_rank_ratings(MANUAL_RATINGS)
if variant_ranking:
    lines = ['| rank | variant | mean manual score | rated images |', '|---:|---|---:|---:|']
    for rank, (variant, score, count) in enumerate(variant_ranking, start=1):
        lines.append(f'| {rank} | `{variant}` | {score:.3f} | {count} |')
    display(Markdown('\n'.join(lines)))
else:
    print('No ratings entered yet.')

if SAVE_MANUAL_RATINGS and rating_rows:
    ratings_json = OUTPUT_DIR / 'manual_ratings.json'
    ratings_csv = OUTPUT_DIR / 'manual_ratings.csv'
    ratings_json.write_text(json.dumps(rating_rows, indent=2, sort_keys=True))
    with ratings_csv.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rating_rows[0]))
        writer.writeheader(); writer.writerows(rating_rows)
    print('Saved:', ratings_json, ratings_csv)


## Finalist validation

After selecting a promising setting, copy its overrides into `FINALIST_OVERRIDES`, expand `SEEDS`, enable `RUN_FINALIST_VALIDATION`, and run the cell below. Compare the finalist against `base_target` and `full_spfc` on every prompt. A useful setting should improve the mean result, not merely rescue one seed or one prompt.


In [ ]:
finalist_records = []
if RUN_FINALIST_VALIDATION:
    finalist_records = run_variant_over_prompts('finalist', FINALIST_OVERRIDES, ALL_PROMPT_KEYS, SEEDS)
    reference_records = run_variant_over_prompts('full_spfc', CORE_ABLATIONS['full_spfc'], ALL_PROMPT_KEYS, SEEDS)
    final_base_records = []
    for seed in SEEDS:
        for prompt_key in ALL_PROMPT_KEYS:
            final_base_records.append(run_base_target(prompt_key, seed))
    show_comparison_matrix([*final_base_records, *reference_records, *finalist_records], ALL_PROMPT_KEYS, ['base_target', 'full_spfc', 'finalist'], save_name='finalist_validation')
    show_diagnostic_table([*reference_records, *finalist_records])
else:
    print('Finalist validation disabled. Fill FINALIST_OVERRIDES and enable RUN_FINALIST_VALIDATION when ready.')


## Recommended interpretation order

1. Verify that ordinary `base_target`, `no_aggregation_control`, and `steering_zero_control` are visually consistent at the same seed. Large differences indicate a sampler/control mismatch that should be resolved before judging SPFC.
2. Compare `full_spfc` with each gate ablation. Inspect target weight and entropy to see whether a switch actually changed arbitration.
3. Sweep `velocity_clip_ratio`; use clamp hit rate to locate the smallest non-destructive bound and identify ranges where the clamp is inactive.
4. Sweep `steering_strength`; remember that it operates after velocity clipping.
5. Sweep `ltp_radius_ratio` and compare LTP retention. If retention is always 1, LTP did not constrain those runs.
6. Test source/target/primitive base weights and temperature only after understanding the safety bounds. These controls interact through softmax.
7. Ablate schedule and primitive count. Early steps usually control structure; late steps have less leverage.
8. Validate finalists over all prompts and multiple seeds, then report per-prompt results rather than only a global average.
